In [2]:
from pathlib import Path
import pandas as pd 
import numpy as np 
import openmatrix as omx

In [9]:
output_root= "C:/Users/USJC173858/temp"
scenarios = {
    "BICOUNTY_MODEL": "../data/external/ccta/nonres",
    "TM-1.6 SW data": "C:/Users/USJC173858/WSP O365/MTC_TM1.7 - Documents/travel-model-one/utilities/trucks/trucks_2026_update/data/interim/matrix_projection/sw_od_trips_with_mtc_format", 
    "TM-1.6": 'C:/temp/mtc_cube_runs/TM-1.6/nonres',
    "TM-1.7_GEN_REEST": 'C:/temp/mtc_cube_runs/TM-1.7_GEN_REEST/nonres',
    "TM-1.7_GEN_NEWSPEC": 'C:/temp/mtc_cube_runs/TM-1.7_GEN_NEWSPEC/nonres',
    "TM-1.7_GEN_IX": 'C:/temp/mtc_cube_runs/TM-1.7_GEN_IX/nonres',
    "TM-1.6_FIX_ROUNDING_ISSUE": "C:/temp/mtc_cube_runs/TM-1.6_FIX_ROUNDING_ISSUE/nonres",
}
path_pattern = "TripsTrk{tod}x.omx"
tods = ["EA", "AM", "MD", "PM", "EV"]

In [10]:
rows = []
for scenario, scenario_path in scenarios.items(): 
    for tod in tods: 
        fname = path_pattern.format(tod=tod)
        ref_path = Path(scenario_path, fname)
        reference_omx = omx.open_file(ref_path, "r")
        for truck_type in reference_omx.list_matrices():
            m = np.array(reference_omx[truck_type])
            internal_break = 1454
            correction_factor = 1
            
            if scenario == "BICOUNTY_MODEL":
                # This is the last TAZ in the 9-county Bay Area Region. 
                # 6274-6594 Are San Joaquin Country trips. 
                # Remanining are model gateways. 
                internal_break = 6272 
                correction_factor = 0 # Avoid county  SanJoaquin-to-SanJoaquin trips. 
                m = m/100 # BICOUNTY_MODEL outputs are multiplied by 100 to avoid rounding issues. 
  
            ii = m[:internal_break][:,:internal_break].sum()
            ix = m[:internal_break][:,internal_break:].sum()
            xi = m[internal_break:][:,:internal_break].sum()
            xx = m[internal_break:][:,internal_break:].sum() * correction_factor
            # print(f"{scenario}_{truck_type}_{tod}: {m.sum():.0f}")
        
            rows.append({
                "scenario": scenario,
                "tod": tod,
                "truck_type": truck_type,
                "type": "internal-internal",
                "truck_trips": ii, 
            })

            rows.append({
                "scenario": scenario,
                "tod": tod,
                "truck_type": truck_type,
                "type": "internal-external",
                "truck_trips": ix, 
            })

            rows.append({
                "scenario": scenario,
                "tod": tod,
                "truck_type": truck_type,
                "type": "external-internal",
                "truck_trips": xi, 
            })

            rows.append({
                "scenario": scenario,
                "tod": tod,
                "truck_type": truck_type,
                "type": "external-external",
                "truck_trips": xx, 
            })
        reference_omx.close()
df = pd.DataFrame(rows)

In [11]:
all_scenarios = ["TM-1.6", "TM-1.6_FIX_ROUNDING_ISSUE","TM-1.6 SW data", "TM-1.7_GEN_REEST", "TM-1.7_GEN_NEWSPEC", "TM-1.7_GEN_IX"]
data_scenarios = ["BICOUNTY_MODEL", "TM-1.6", "TM-1.6_FIX_ROUNDING_ISSUE","TM-1.6 SW data"]

print ("TABLE 1. Trip Generation")
df.pivot_table(
    index = ["truck_type", "type"], 
    columns = ["scenario"], 
    values = "truck_trips", 
    aggfunc = "sum"
)[data_scenarios].style.format("{:,.0f}")

TABLE 1. Trip Generation


In [123]:
print ("TABLE 1. Trip Generation")
df[df["type"] == "internal-internal"].pivot_table(
    index = ["truck_type","scenario" ], 
    columns = ["tod"], 
    values = "truck_trips", 
    aggfunc = "sum"
)[["EA", "AM","MD","PM", "EV"]].style.format("{:,.0f}")

TABLE 1. Trip Generation


In [36]:
print ("TABLE 1. Trip Generation")
df.pivot_table(
    index = ["truck_type","scenario" ], 
    columns = ["tod"], 
    values = "truck_trips", 
    aggfunc = "sum"
)[["EA", "AM","MD","PM", "EV"]].style.format("{:,.0f}")

TABLE 1. Trip Generation


In [10]:
# sW raw data 
path = "../data/interim/matrix_projection/SwTLN_to_TMTaz_TRIPS_FFM_2020.omx"
omx_file = omx.open_file(path, 'r')
truck_type_map = {"HT": "Heavy", "MT": "Medium", "LT": "Light"}
rows = []
for i in omx_file.list_matrices():
    m = np.array(omx_file[i])
    ii = m[:1454,:1454].sum()
    ix = m[:1454,1454:].sum()
    xi = m[1454:,:1454].sum()
    xx = m[1454:,1454:].sum()
    total = m.sum()
    truck_type = truck_type_map[i[:2]]
    rows.append({
        "truck_type": truck_type , 
        "internal-internal": ii, 
        "internal-external": ix,
        "external-internal": xi,
        "external-external": xx, 
        "total": total
        })
summary = pd.DataFrame(rows)
summary.sum()

truck_type           HeavyHeavyHeavyHeavyHeavyHeavyHeavyHeavyHeavyH...
internal-internal                                          3193.199219
internal-external                                           428.512482
external-internal                                           427.899597
external-external                                                  0.0
total                                                       4049.61084
dtype: object

In [4]:
# sW raw data 
path = "../data/interim/matrix_projection/SwTAZ_to_TMTaz_TRIPS_FFM_2020.omx"
omx_file = omx.open_file(path, 'r')
truck_type_map = {"HT": "Heavy", "MT": "Medium", "LT": "Light"}
rows = []
for i in omx_file.list_matrices():
    m = np.array(omx_file[i])
    ii = m[:1454,:1454].sum()
    ix = m[:1454,1454:].sum()
    xi = m[1454:,:1454].sum()
    xx = m[1454:,1454:].sum()
    total = m.sum()
    truck_type = truck_type_map[i[:2]]
    rows.append({
        "truck_type": truck_type , 
        "internal-internal": ii, 
        "internal-external": ix,
        "external-internal": xi,
        "external-external": xx, 
        "total": total
        })
summary = pd.DataFrame(rows)
summary.sum()

truck_type           HeavyHeavyHeavyHeavyHeavyHeavyHeavyHeavyHeavyH...
internal-internal                                             537895.0
internal-external                                         38822.136719
external-internal                                         43734.042969
external-external                                                  0.0
total                                                        620451.25
dtype: object

In [5]:
# sW raw data 
path = "../data/interim/matrix_projection/SwTazTln_to_TMTaz_TRIPS_FFM_2020.omx"
omx_file = omx.open_file(path, 'r')
truck_type_map = {"HT": "Heavy", "MT": "Medium", "LT": "Light"}
rows = []
for i in omx_file.list_matrices():
    m = np.array(omx_file[i])
    ii = m[:1454,:1454].sum()
    ix = m[:1454,1454:].sum()
    xi = m[1454:,:1454].sum()
    xx = m[1454:,1454:].sum()
    total = m.sum()
    truck_type = truck_type_map[i[:2]]
    rows.append({
        "truck_type": truck_type , 
        "internal-internal": ii, 
        "internal-external": ix,
        "external-internal": xi,
        "external-external": xx, 
        "total": total
        })
summary = pd.DataFrame(rows)
summary.sum()

truck_type           HeavyHeavyHeavyHeavyHeavyHeavyHeavyHeavyHeavyH...
internal-internal                                             537895.0
internal-external                                         39250.648438
external-internal                                           44161.9375
external-external                                                  0.0
total                                                      621307.6875
dtype: object

In [8]:
import openmatrix as omx
import pandas as pd
import numpy as np

In [22]:
fpath = "../data/interim/cube_io/statewide_od_matrices/TRIPS_FFM_2020.omx"  
omx_file = omx.open_file(fpath, 'r')

crosswalk = pd.read_csv("../data/interim/matrix_projection/crosswalk.csv")

In [34]:
m = np.array(omx_file["MT1_FR_EXT_OFF"])

In [43]:
m[6988].sum()

np.float64(0.0859)

In [26]:
m[m>0].sum()

np.float64(17816.405600000002)

In [15]:
m[m<0].sum()

np.float64(-24.203999999999994)

In [21]:
np.maximum(m, 0).sum()

np.float64(3452.3965000000003)

In [21]:
zone_ids = crosswalk[crosswalk["type"].isin(["internal_gate", "internal_zone"])]["from_zone_id"].unique() - 1
n = 7000
mask = np.zeros((n, n), dtype=bool)
mask[zone_ids, :] = True   # rows
mask[:, zone_ids] = True   # columns

In [22]:
internal_trips = 0
total = 0 
for name in omx_file.list_matrices():
    m = np.array(omx_file[name])
    internal_trips += m[zone_ids][:,zone_ids].sum()
    total += m[mask].sum()
print(f"Total: {total:,.0f} \n Internal_trips: {internal_trips:,.0f} \n External Trips: {(total -internal_trips):,.0f}")

Total: 621,310 
 Internal_trips: 537,891 
 External Trips: 83,420


In [129]:
internal_trips = 0
total = 0 
for name in omx_file.list_matrices():
    m = np.array(omx_file[name])
    internal_trips += m[zone_ids][:,zone_ids].sum()
    total += m[mask].sum()

print(f"Total: {total:,.0f} \n Internal_trips: {internal_trips:,.0f} \n External Trips: {(total -internal_trips):,.0f}")

Total: 4,044 
 Internal_trips: 6 
 External Trips: 4,038


In [120]:
m[internal_zones_ids][:,internal_zones_ids]

array([[0.   , 0.   , 0.   , ..., 0.   , 0.   , 0.   ],
       [0.   , 0.   , 0.   , ..., 0.   , 0.   , 0.   ],
       [0.   , 0.   , 0.   , ..., 0.   , 0.   , 0.   ],
       ...,
       [0.   , 0.   , 0.   , ..., 0.008, 0.008, 0.   ],
       [0.   , 0.   , 0.   , ..., 0.008, 0.01 , 0.   ],
       [0.   , 0.   , 0.   , ..., 0.   , 0.   , 0.   ]],
      shape=(1141, 1141))

In [114]:
internal_zones_ids

array([ 783,  784,  785, ..., 6704, 6705, 6706], shape=(1141,))